# Tamakkan, BDD100K to YOLO Conversion

This notebook converts the BDD100K detection annotations into the YOLO format we use for training. It runs in three cells.

1. **Main conversion.** Reads the BDD100K per-image JSON labels, drops the `train` class, merges `bike`, `rider`, and `motor` into a single `vulnerable_road_user` class, converts bounding boxes to YOLO's normalized `(cx, cy, w, h)` format, writes one `.txt` label file per image, and produces a `data.yaml`.
2. **Rewrite `data.yaml` with relative paths.** Produces a clean `data.yaml` using relative image paths under the output dataset root, then re-prints the conversion summary.
3. **Rewrite `data.yaml` with absolute paths.** Final version that points directly at the original BDD100K image folder, so we do not have to duplicate the 70k+ images into a new directory. This is the `data.yaml` the training notebook actually loads.

Final class mapping (7 classes):

| ID | Name |
|---|---|
| 0 | car |
| 1 | truck |
| 2 | bus |
| 3 | person |
| 4 | traffic light |
| 5 | traffic sign |
| 6 | vulnerable_road_user (bike + rider + motor merged) |

In [ ]:
import json
import os
from pathlib import Path

In [ ]:
# paths
IMAGES_ROOT = r"C:\Users\Admin\Desktop\Grad Project\Code\Datasets\bdd100k_images_100k\100k"
LABELS_ROOT = r"C:\Users\Admin\Desktop\Grad Project\Code\Datasets\bdd100k_labels\100k"
OUTPUT_ROOT = r"C:\Users\Admin\Desktop\Grad Project\Code\Datasets\BDD100k_V2"

In [ ]:
# classes we drop entirely from the dataset
IGNORE_CLASSES = {"train"}

# classes we merge together into one vulnerable_road_user class
MERGE_INTO_VRU = {"bike", "rider", "motor"}

# final class ID mapping used by the YOLO labels
CLASS_MAP = {
    "car":                  0,
    "truck":                1,
    "bus":                  2,
    "person":               3,
    "traffic light":        4,
    "traffic sign":         5,
    "vulnerable_road_user": 6,  # bike + rider + motor get merged here
}

# BDD100K standard image dimensions, used to normalize bbox coordinates
IMAGE_WIDTH  = 1280
IMAGE_HEIGHT = 720

SPLITS = ["train", "val", "test"]

In [ ]:
def convert_bbox_to_yolo(x1, y1, x2, y2, img_w, img_h):
    """ Convert BDD100K absolute (x1, y1, x2, y2) bbox into YOLO normalized (cx, cy, w, h) bbox. """
    cx = (x1 + x2) / 2.0 / img_w
    cy = (y1 + y2) / 2.0 / img_h
    w  = (x2 - x1) / img_w
    h  = (y2 - y1) / img_h
    # clamp to [0, 1] in case of edge annotations
    cx = max(0.0, min(1.0, cx))
    cy = max(0.0, min(1.0, cy))
    w  = max(0.0, min(1.0, w))
    h  = max(0.0, min(1.0, h))
    return cx, cy, w, h


def process_split(split: str, stats: dict):
    """ Process one split (train, val, or test) and write YOLO labels. """
    labels_dir = Path(LABELS_ROOT) / split
    output_labels_dir = Path(OUTPUT_ROOT) / "labels" / split
    output_labels_dir.mkdir(parents=True, exist_ok=True)

    json_files = list(labels_dir.glob("*.json"))
    print(f"\n[{split}] Found {len(json_files)} JSON label files")

    for json_path in json_files:
        with open(json_path, "r") as f:
            data = json.load(f)

        objects = []
        if "frames" in data:
            for frame in data["frames"]:
                objects.extend(frame.get("objects", []))
        elif "objects" in data:
            objects = data["objects"]

        yolo_lines = []

        for obj in objects:
            category = obj.get("category", "").strip().lower()
            box2d    = obj.get("box2d", None)

            # no bounding box, skip
            if box2d is None:
                stats["no_box"] += 1
                continue

            # dropped classes
            if category in IGNORE_CLASSES:
                stats["ignored"] += 1
                continue

            # merge VRU classes into one
            if category in MERGE_INTO_VRU:
                class_id = CLASS_MAP["vulnerable_road_user"]
                stats["vru_merged"] += 1
            elif category in CLASS_MAP:
                class_id = CLASS_MAP[category]
            else:
                # any class we did not expect, track it but skip the annotation
                stats["unknown"].add(category)
                stats["unknown_count"] += 1
                continue

            x1 = box2d["x1"]
            y1 = box2d["y1"]
            x2 = box2d["x2"]
            y2 = box2d["y2"]

            # zero-area or inverted boxes
            if x2 <= x1 or y2 <= y1:
                stats["degenerate"] += 1
                continue

            cx, cy, w, h = convert_bbox_to_yolo(x1, y1, x2, y2, IMAGE_WIDTH, IMAGE_HEIGHT)
            yolo_lines.append(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
            stats["total_annotations"] += 1
            stats["per_class"][class_id] += 1

        # write the .txt label file. 
        txt_filename = json_path.stem + ".txt"
        with open(output_labels_dir / txt_filename, "w") as f:
            f.write("\n".join(yolo_lines))

        stats["images_processed"] += 1

    print(f"[{split}] Done. Written to: {output_labels_dir}")


def write_data_yaml():
    """ Write the initial data.yaml for YOLO training. """
    yaml_content = f"""# Tamakkan, YOLOv11s dataset config
# 7 classes, with bike + rider + motor merged into vulnerable_road_user

path: {OUTPUT_ROOT}
train: images/train
val:   images/val
test:  images/test

nc: 7
names:
  0: car
  1: truck
  2: bus
  3: person
  4: traffic light
  5: traffic sign
  6: vulnerable_road_user
"""
    yaml_path = Path(OUTPUT_ROOT) / "data.yaml"
    yaml_path.parent.mkdir(parents=True, exist_ok=True)
    with open(yaml_path, "w") as f:
        f.write(yaml_content)
    print(f"\ndata.yaml written to: {yaml_path}")


def print_summary(stats: dict):
    class_names = {
        0: "car", 1: "truck", 2: "bus", 3: "person",
        4: "traffic light", 5: "traffic sign", 6: "vulnerable_road_user"
    }
    print("\nCONVERSION SUMMARY")
    print(f"  Images processed      : {stats['images_processed']:,}")
    print(f"  Total annotations     : {stats['total_annotations']:,}")
    print(f"  VRU merged (b+r+m)    : {stats['vru_merged']:,}")
    print(f"  Ignored (train class) : {stats['ignored']:,}")
    print(f"  Skipped (no box)      : {stats['no_box']:,}")
    print(f"  Skipped (degenerate)  : {stats['degenerate']:,}")
    print(f"  Unknown classes       : {stats['unknown_count']:,} -> {stats['unknown']}")
    print("\n  Per-class annotation counts:")
    for cid, count in sorted(stats["per_class"].items()):
        print(f"    [{cid}] {class_names[cid]:<22}: {count:,}")

In [ ]:
# run the conversion

if __name__ == "__main__":
    stats = {
        "images_processed":   0,
        "total_annotations":  0,
        "vru_merged":         0,
        "ignored":            0,
        "no_box":             0,
        "degenerate":         0,
        "unknown_count":      0,
        "unknown":            set(),
        "per_class":          {i: 0 for i in range(7)},
    }

    print("Tamakkan BDD100K to YOLO conversion")
    print(f"Output: {OUTPUT_ROOT}\n")

    # this script only converts labels. the original BDD100K images are left where they are.
    # data.yaml is updated in a later cell to point at the original image folder so we do not duplicate 70k+ files.

    for split in SPLITS:
        process_split(split, stats)

    write_data_yaml()
    print_summary(stats)

    print("\nupdate data.yaml image paths, then run training.")
    print("Images are not copied. Update the path in data.yaml to point to the original image folder.")

In [ ]:
OUTPUT_ROOT = r"C:\Users\Admin\Desktop\Grad Project\Code\Datasets\BDD100k_V2"

# clean version of data.yaml using paths relative to OUTPUT_ROOT
yaml_content = """# Tamakkan, YOLOv11s dataset config
# 7 classes, with bike + rider + motor merged into vulnerable_road_user

train: images/train
val:   images/val
test:  images/test

nc: 7
names:
  0: car
  1: truck
  2: bus
  3: person
  4: traffic light
  5: traffic sign
  6: vulnerable_road_user
"""

yaml_path = Path(OUTPUT_ROOT) / "data.yaml"
yaml_path.parent.mkdir(parents=True, exist_ok=True)
with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(yaml_content)

print(f"data.yaml written to: {yaml_path}")
print("CONVERSION SUMMARY")
print(f"  Images processed      : {stats['images_processed']:,}")
print(f"  Total annotations     : {stats['total_annotations']:,}")
print(f"  VRU merged (b+r+m)    : {stats['vru_merged']:,}")
print(f"  Ignored (train class) : {stats['ignored']:,}")
print(f"  Unknown classes       : {stats['unknown_count']:,} -> {stats['unknown']}")

class_names = {0:"car",1:"truck",2:"bus",3:"person",4:"traffic light",5:"traffic sign",6:"vulnerable_road_user"}
print("\n  Per-class annotation counts:")
for cid, count in sorted(stats["per_class"].items()):
    print(f"    [{cid}] {class_names[cid]:<22}: {count:,}")

In [ ]:
OUTPUT_ROOT = r"C:\Users\Admin\Desktop\Grad Project\Code\Datasets\BDD100k_V2"

# final data.yaml. uses absolute paths to the original BDD100K image folder so we do not have to duplicate the 70k+ images into the output directory.
yaml_content = """# Tamakkan, YOLOv11s dataset config
# 7 classes, with bike + rider + motor merged into vulnerable_road_user

"""

yaml_path = Path(OUTPUT_ROOT) / "data.yaml"
with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(yaml_content)

print(f"data.yaml updated at: {yaml_path}")